# 🚗 Day 06a: HLD — Uber / Ride Sharing

---

## 🎯 What You'll Master Today
- Location tracking & updates
- Driver-rider matching algorithm
- ETA calculation
- Surge pricing
- Geohash / Quadtree for spatial indexing

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  UBER / RIDE SHARING — ARCHITECTURE                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Core Flow:                                                      ║
║  1. Rider requests ride (pickup, destination)                    ║
║  2. System finds nearby available drivers                        ║
║  3. Match & notify driver                                        ║
║  4. Driver accepts → ETA to pickup                               ║
║  5. Trip in progress → real-time tracking                        ║
║  6. Trip ends → calculate fare → payment                        ║
║                                                                   ║
║  ┌──────┐  request  ┌──────────┐  notify  ┌──────┐            ║
║  │Rider │──────────►│ Matching │─────────►│Driver│            ║
║  │ App  │           │ Service  │          │ App  │            ║
║  └──┬───┘           └────┬─────┘          └──┬───┘            ║
║     │                    │                    │                  ║
║     │  ┌─────────────────▼────────────────┐   │                ║
║     │  │       Location Service           │   │                ║
║     └──┤  (track all driver positions)    ├───┘                ║
║        │  Update every 3-4 seconds        │                     ║
║        └────────────┬─────────────────────┘                     ║
║                     │                                            ║
║              ┌──────▼───────┐                                    ║
║              │  Spatial     │                                    ║
║              │  Index       │   (Geohash / Quadtree)            ║
║              │  (find nearby)│                                   ║
║              └──────────────┘                                    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Scale Estimation
# ============================================================

print("""
📊 UBER SCALE ESTIMATION

Assumptions:
  • 5M active drivers
  • 20M rides/day
  • Driver sends location every 3 seconds

Location Update QPS:
  5M drivers × (1/3 sec) ≈ 1.7M updates/sec (!!)
  This is the HOTTEST path in the system

Ride Request QPS:
  20M / 86400 ≈ 230 requests/sec (manageable)

Location data per update:
  {driver_id, lat, lng, timestamp, speed, heading} ≈ 100 bytes
  1.7M × 100B = 170 MB/sec ingress (location data alone)

Storage:
  Location history: 1.7M/sec × 86400 × 100B ≈ 14.7 TB/day
  → Keep raw for 24h, aggregate for analytics

💡 Key insight: Location updates are the hardest part.
   Need in-memory spatial index, not disk-based DB.
""")

In [ ]:
# ============================================================
# 2. Spatial Indexing — Geohash
# ============================================================

print("""
🗺️ GEOHASH — Encoding Location into a String

Idea: Divide world into grid cells, each with a unique string.
      Nearby locations share a common prefix!

  Precision levels:
  ┌──────────────────────────────────────────┐
  │ Length │ Cell Size      │ Use Case        │
  ├──────────────────────────────────────────┤
  │ 1      │ 5,000km × 5,000km│ Continent    │
  │ 3      │ 156km × 156km    │ Country/State│
  │ 4      │ 39km × 19.5km    │ City         │
  │ 5      │ 4.9km × 4.9km    │ Neighborhood │
  │ 6      │ 1.2km × 0.6km    │ ~Block       │
  │ 7      │ 153m × 153m      │ Street       │
  │ 8      │ 38m × 19m        │ Building     │
  └──────────────────────────────────────────┘

  Example:
  NYC Times Square: "dr5ru7" (geohash length 6)
  Nearby locations also start with "dr5ru..."

  For Uber: Use geohash length 6 (~1km² cells)
  Store: geohash → set of {driver_ids} in Redis
  Query: "find drivers in dr5ru7 and adjacent cells"
""")

# Simple geohash implementation
BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'

def geohash_encode(lat, lon, precision=6):
    lat_range = [-90.0, 90.0]
    lon_range = [-180.0, 180.0]
    bits = []
    is_lon = True  # alternate between lon and lat
    
    while len(bits) < precision * 5:
        if is_lon:
            mid = (lon_range[0] + lon_range[1]) / 2
            if lon >= mid:
                bits.append(1)
                lon_range[0] = mid
            else:
                bits.append(0)
                lon_range[1] = mid
        else:
            mid = (lat_range[0] + lat_range[1]) / 2
            if lat >= mid:
                bits.append(1)
                lat_range[0] = mid
            else:
                bits.append(0)
                lat_range[1] = mid
        is_lon = not is_lon
    
    # Convert bits to base32
    result = []
    for i in range(0, len(bits), 5):
        chunk = bits[i:i+5]
        val = sum(b << (4-j) for j, b in enumerate(chunk))
        result.append(BASE32[val])
    
    return ''.join(result)

# Demo
locations = [
    ("Times Square, NYC", 40.7580, -73.9855),
    ("Central Park, NYC", 40.7829, -73.9654),
    ("Brooklyn Bridge",   40.7061, -73.9969),
    ("San Francisco",     37.7749, -122.4194),
]

for name, lat, lon in locations:
    gh = geohash_encode(lat, lon)
    print(f"{name:25s} → {gh}")

print("\n💡 Notice: NYC locations share prefix 'dr5r', SF is completely different")

In [ ]:
# ============================================================
# 3. Driver Matching Algorithm
# ============================================================

import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """Distance in km between two GPS points."""
    R = 6371  # Earth radius in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2 + 
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * 
         math.sin(dlon/2)**2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


class MatchingService:
    def __init__(self, search_radius_km=3):
        self.drivers = {}  # driver_id → {lat, lon, available}
        self.search_radius = search_radius_km
    
    def update_location(self, driver_id, lat, lon):
        self.drivers[driver_id] = {"lat": lat, "lon": lon, "available": True}
    
    def find_nearest(self, rider_lat, rider_lon, top_k=5):
        """Find top-K nearest available drivers."""
        candidates = []
        for did, info in self.drivers.items():
            if not info["available"]:
                continue
            dist = haversine_distance(rider_lat, rider_lon, info["lat"], info["lon"])
            if dist <= self.search_radius:
                candidates.append((did, dist))
        
        candidates.sort(key=lambda x: x[1])
        return candidates[:top_k]
    
    def match_ride(self, rider_lat, rider_lon):
        """Match and assign nearest driver."""
        nearest = self.find_nearest(rider_lat, rider_lon, top_k=1)
        if not nearest:
            return None
        driver_id, dist = nearest[0]
        self.drivers[driver_id]["available"] = False
        return {"driver": driver_id, "distance_km": round(dist, 2),
                "eta_min": round(dist / 0.5, 1)}  # assume 30 km/h city speed

# --- Demo ---
svc = MatchingService(search_radius_km=5)

# Simulate drivers around NYC
import random
random.seed(42)
for i in range(20):
    lat = 40.75 + random.uniform(-0.03, 0.03)
    lon = -73.98 + random.uniform(-0.03, 0.03)
    svc.update_location(f"driver_{i}", lat, lon)

# Rider at Times Square
rider_lat, rider_lon = 40.7580, -73.9855
print("Top 5 nearest drivers:")
for did, dist in svc.find_nearest(rider_lat, rider_lon):
    print(f"  {did}: {dist:.2f} km away")

print(f"\nMatched: {svc.match_ride(rider_lat, rider_lon)}")

In [ ]:
# ============================================================
# 4. Surge Pricing & Full Architecture
# ============================================================

print("""
💰 SURGE PRICING

  demand_supply_ratio = ride_requests_in_area / available_drivers_in_area
  
  Ratio    │ Multiplier │ Meaning
  < 1.0    │ 1.0×       │ Normal pricing
  1.0-2.0  │ 1.2-1.5×   │ Slightly busy
  2.0-3.0  │ 1.5-2.0×   │ Busy
  > 3.0    │ 2.0-3.0×   │ Very high demand
  
  Computed per geohash cell, recalculated every ~2 minutes
  Goal: incentivize more drivers to the area
""")

print("""
╔════════════════════════════════════════════════════════════════╗
║  UBER FULL ARCHITECTURE                                      ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Services:                                                    ║
║  ┌────────────────┐    ┌─────────────┐                       ║
║  │ Location Svc   │    │ Matching Svc│                       ║
║  │ (1.7M upd/sec) │    │ (find nearby│                       ║
║  │ Redis + Geohash│    │  + assign)  │                       ║
║  └────────────────┘    └─────────────┘                       ║
║                                                                ║
║  ┌────────────────┐    ┌─────────────┐                       ║
║  │ Trip Svc       │    │ Pricing Svc │                       ║
║  │ (state machine │    │ (base fare  │                       ║
║  │  REQUESTED →   │    │  + distance │                       ║
║  │  MATCHED →     │    │  + time     │                       ║
║  │  IN_PROGRESS → │    │  + surge)   │                       ║
║  │  COMPLETED)    │    │             │                       ║
║  └────────────────┘    └─────────────┘                       ║
║                                                                ║
║  ┌────────────────┐    ┌─────────────┐                       ║
║  │ ETA Svc        │    │ Payment Svc │                       ║
║  │ (graph algo    │    │ (Stripe/    │                       ║
║  │  + real-time   │    │  in-app     │                       ║
║  │  traffic data) │    │  wallet)    │                       ║
║  └────────────────┘    └─────────────┘                       ║
║                                                                ║
║  Data Stores:                                                  ║
║  • Redis: real-time driver locations (geohash → driver set)  ║
║  • Cassandra: trip history, location traces                  ║
║  • PostgreSQL: users, drivers, payments                      ║
║  • Kafka: async events (trip updates, analytics, surge calc) ║
║                                                                ║
║  Communication:                                                ║
║  • Driver app: WebSocket (real-time location + ride offers)  ║
║  • Rider app: WebSocket (ETA updates, driver tracking)       ║
║  • Push notifications for ride status changes                ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design Uber | Location svc (Redis+geohash), matching, trip state machine, surge pricing |
| 2 | How to find nearby drivers? | Geohash or Quadtree. Redis: geohash → driver set. Check cell + neighbors |
| 3 | Geohash vs Quadtree? | Geohash: string prefix, easy in Redis. Quadtree: better for uneven density |
| 4 | How surge pricing works? | demand/supply ratio per cell. Recompute every 2 min via Kafka |
| 5 | Handle 1.7M location updates/sec? | Redis in-memory. Partition by geohash region. Async persistence to Cassandra |
| 6 | ETA calculation? | Graph (road network) + real-time traffic. Shortest path with weights |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Location updates (1.7M/sec) is the hardest part       │
## │  • Geohash: encode lat/lng → string. Nearby = same prefix│
## │  • Redis: geohash cell → set of driver_ids               │
## │  • Matching: nearest K drivers → notify first available  │
## │  • Surge: demand/supply ratio per geohash cell           │
## │  • Trip = state machine (requested→matched→active→done) │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **System Design Interview Framework**